In [ ]:
import pickle

with open('../I001_Results/DATA_PICK_010_A.pkl', 'rb') as f:
    data = pickle.load(f)

data.keys()


In [ ]:
import json
import numpy as np


def compute_mesh_porosity(mesh_file):
    """
    Compute porosity from only the mesh nodes and element connectivity.

    The function does not use mesh metadata such as geometry, parameters,
    summary, hole centers, hole radii, or saved porosity values.

    porosity = void_area / total_area
    void_area = total_area - solid_mesh_area
    """
    with open(mesh_file, "r") as f:
        mesh = json.load(f)

    nodes_raw = mesh["nodes"]
    elements = mesh["elements"]

    nodes = np.array([
        (node["x"], node["y"]) if isinstance(node, dict) else (node[0], node[1])
        for node in nodes_raw
    ], dtype=float)

    def polygon_area(points):
        x = points[:, 0]
        y = points[:, 1]
        return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

    solid_area = 0.0
    n_nodes = len(nodes)

    for element in elements:
        element = [int(node_id) for node_id in element]

        # Mesh JSON files in this project use 0-based connectivity. This
        # fallback keeps the function useful for 1-based mesh exports too.
        if element and min(element) >= 1 and max(element) == n_nodes:
            element = [node_id - 1 for node_id in element]

        clean_element = []
        for node_id in element:
            if node_id not in clean_element:
                clean_element.append(node_id)

        if len(clean_element) < 3:
            continue

        solid_area += polygon_area(nodes[clean_element])

    xmin, ymin = nodes.min(axis=0)
    xmax, ymax = nodes.max(axis=0)
    total_area = (xmax - xmin) * (ymax - ymin)
    void_area = total_area - solid_area
    porosity = void_area / total_area

    return {
        "porosity": float(porosity),
        "void_area": float(void_area),
        "solid_area": float(solid_area),
        "total_area": float(total_area),
    }


compute_mesh_porosity("../C001_Mesh_files/R002.mesh.json")
